# 2.9 FA 的演进：GQA、MLA、SFA

2.3 节已见过 GQA / MQA 与 KV Cache 的概念，本节算清它们的**经济账**，再补两位重量级成员：DeepSeek 的 **MLA**（把 KV Cache 再砍一个数量级）与稀疏路线 **SFA**（把计算量砍两个数量级）。它们正是昇腾 `FusedInferAttentionScore` 等算子接口里那些“奇怪参数”的真正来源。

先立靶子——一切演进的驱动力是**推理时代的 KV Cache 与长上下文**。

## 一、靶子：KV Cache 的经济账

回顾 KV Cache 字节数公式（2.3 节）：

**KV Cache = 2（K 和 V）× 层数 × KV 头数 × d_head × 序列长度 × 精度**

In [1]:
def kv_cache_gb(n_layers, n_kv_heads, d_head, seq_len, bytes_per_elem=2):
    return 2 * n_layers * n_kv_heads * d_head * seq_len * bytes_per_elem / 1e9

print('KV Cache 账单（Llama-3-8B 同级：32 层、d_head=128、fp16）：')
for seq in [4096, 32768, 131072]:
    gqa = kv_cache_gb(32, 8, 128, seq)
    mha = kv_cache_gb(32, 32, 128, seq)
    print(f'  序列 {seq:6d}：MHA = {mha:7.2f} GB   GQA(8 KV 头) = {gqa:7.2f} GB（省 75%）')

KV Cache 账单（Llama-3-8B 同级：32 层、d_head=128、fp16）：
  序列   4096：MHA =    2.15 GB   GQA(8 KV 头) =    0.54 GB（省 75%）
  序列  32768：MHA =   17.18 GB   GQA(8 KV 头) =    4.29 GB（省 75%）
  序列 131072：MHA =   68.72 GB   GQA(8 KV 头) =   17.18 GB（省 75%）

128k 序列下 MHA 要 68 GB——比模型本身还大。这就是演进的全部动力：**让每个 token 的 KV 更小（GQA、MLA），或者干脆不算大部分 token（SFA）**。

## 二、MHA → MQA → GQA：共享 KV 头

![Attention 变体](images/attention_variants.svg)

在 2.3 节的概念之外，补上工程视角的三个要点：

1. GQA 分组共享 KV 后，**同组各头的 Q 拼成大矩阵一次算完**——FA 算子的单头负载从 (n, d) 变成 (g·n, d)，正是推理算子最典型的工作负载；
2. 算子接口里的 `num_heads / num_kv_heads`（Q 头数 / KV 头数）就是这两层拓扑的参数化；
3. MQA 已基本弃用（质量损失明显）；GQA 是质量与显存的甜点（Llama2-70B、Llama3 全系采用）。

## 三、MLA：低秩压缩 + RoPE 解耦（DeepSeek）

GQA 只是把头分组，MLA 更激进：**根本不为每个头保存 K/V，只保存一个压缩向量**。

![MLA 架构](images/mla_architecture.svg)

**压缩通路**：h_t（d_model = 7168 维）先下投影成 **c_t（512 维）**——这就是全部要缓存的“潜在 KV”；使用时按头上投影展开成 k_i / v_i（各 128 维）。

**RoPE 解耦**：RoPE 是旋转运算，与“先压缩再展开”**不可交换**。MLA 的处理是把 K 拆成两段拼接：位置无关的 512 维走压缩通路；位置相关的 **64 维单独投影、单独缓存**。

In [1]:
d_model, d_c, d_h, n_h = 7168, 512, 128, 128
mha_per_token = 2 * n_h * d_h                 # 每层每 token 缓存的元素个数
mla_per_token = d_c + 64                      # c_KV(512) + RoPE 通道(64)
print(f'DeepSeek-V2 同级，每层每 token 缓存元素数：')
print(f'  MHA：2×{n_h}×{d_h} = {mha_per_token}')
print(f'  MLA：{d_c} + 64 = {mla_per_token}（压缩 {mha_per_token/mla_per_token:.0f} 倍）')
layers, seq, byte = 60, 131072, 1             # fp8
print(f'  60 层 × 128k 序列 × fp8：MHA ≈ {mha_per_token*layers*seq*2/1e9:.0f} GB，'
      f'MLA ≈ {mla_per_token*layers*seq*byte/1e9:.1f} GB')

DeepSeek-V2 同级，每层每 token 缓存元素数：
  MHA：2×128×128 = 32768
  MLA：512 + 64 = 576（压缩 57 倍）
  60 层 × 128k 序列 × fp8：MHA ≈ 515 GB，MLA ≈ 4.5 GB

**权重吸收**：推理时不必显式展开 k_i——把上投影矩阵“吸收”进 Q 侧，点积直接打在 512 维的 c_t 上：

In [1]:
import numpy as np
np.random.seed(0)

c = np.random.randn(d_c)                      # 压缩后的潜在向量
W_UK = np.random.randn(d_h, d_c)              # 上投影（nope 部分）
q = np.random.randn(d_h)                      # 某个头的查询向量

lhs = q @ (W_UK @ c)                          # 先展开 k = W_UK·c，再点积
rhs = (q @ W_UK) @ c                          # 把 W_UK 吸收进 q，直接对 c 点积
print('权重吸收恒等式  q·(W·c) == (q·W)·c ：', np.allclose(lhs, rhs))
print('展开路径：先算 k（%d 维），再点积' % d_h)
print('吸收路径：q 先乘 W_UK，点积直接打在 %d 维的 c 上 —— 无需展开任何 per-head 的 k' % d_c)

权重吸收恒等式  q·(W·c) == (q·W)·c ： True
展开路径：先算 k（128 维），再点积
吸收路径：q 先乘 W_UK，点积直接打在 512 维的 c 上 —— 无需展开任何 per-head 的 k

**给算子开发的含义**（本课程重点）：

- MLA 的“压缩-展开”两段结构与标准 GQA 完全不同：算子必须支持**独立的 rope 通道输入**（`FusedInferAttentionScore` 的 `query_rope / key_rope` 参数），以及**压缩向量直接参与 GEMM** 的新形态；
- 这类新形态也带来新的调度难题：Q 侧要展开、KV 侧是压缩的，S1/S2 切分不再对称（第 9 章负载均衡的难点之一）。

## 四、SFA：Sparse FlashAttention——不算大多数 token

![SFA 两阶段](images/sfa_two_stage.svg)

GQA / MLA 省的是**显存与带宽**，计算量仍是 O(n²)。SFA 路线更狠：**先选出 ~2% 的关键 token，只对它们做精确 Attention**。两阶段（DeepSeek V3.2 的 DSA、GLM-5 同路线，昇腾已提供对应融合算子）：

1. **Lightning Indexer**：q、k 低维投影打分 → ReLU → ReduceSum → **TopK** 选出关键 token 索引；
2. **Sparse FlashAttention**：按索引离散搬运选中的 KV，再做标准的 Online Softmax 分块计算。

In [1]:
print('SFA 计算量（固定选 TopK=2048 个 token，对比 dense）：')
for n in [8192, 32768, 131072, 524288]:
    print(f'  上下文 {n:7d}：稀疏部分计算量 ≈ dense 的 {2048/n:6.1%}')

SFA 计算量（固定选 TopK=2048 个 token，对比 dense）：
  上下文    8192：稀疏部分计算量 ≈ dense 的  25.0%
  上下文   32768：稀疏部分计算量 ≈ dense 的   6.2%
  上下文  131072：稀疏部分计算量 ≈ dense 的   1.6%
  上下文  524288：稀疏部分计算量 ≈ dense 的   0.4%

注意：这与 2.5 节因果掩码“整块跳过”一脉相承——**从按规则跳过（下三角结构），进化到学习着跳过（TopK）**；且对选中的 token，计算是**精确**的。

**昇腾上的工程挑战**（本课程高阶主题的来源）：

- **离散访存**：选中的 token 在 KV Cache 里七零八落，单次搬运小、指令下发频繁——**指令下发的耗时甚至可能超过数据搬运本身**。对策：利用 Cube:Vector = 1:2 的核配比让 Vector 核发射访存指令；离散数据两两聚合（srcGap 参数）；
- Indexer 里的 ReLU 用 Cube 核 FixP 组件**随路计算**、TopK 做成融合算子——不占独立 kernel；
- 两阶段间的依赖与负载均衡 → 第 9 章。

## 五、族谱总览

| 方案 | KV Cache | 计算量 | 质量 | 代表模型 | 昇腾算子要点 |
|--|--|--|--|--|--|
| MHA | 100%（基准） | O(n²) | 基准 | Llama2-7B | — |
| GQA | g/h（如 25%） | O(n²) | 几乎无损 | Llama2-70B、Llama3 | num_kv_heads 参数 |
| MQA | 1/h | O(n²) | 明显掉点 | （已少用） | — |
| MLA | ~1/57 | O(n²)（常数更优） | 无损（重构） | DeepSeek-V2/V3 | query_rope / key_rope 独立入参 |
| SFA | 同左（+索引） | **k/n（~2%）** | 近似无损 | DeepSeek-V3.2、GLM-5 | LightningIndexer + 聚合离散访存 |

## 章节测验

1.（填空）KV Cache 字节数 = ____ × 层数 × KV 头数 × d_head × 序列长度 × 精度。
2. GQA 相比 MHA 省的是什么资源？为什么说它还“顺便”改变了 FA 算子的工作负载形状？
3. MLA 为什么必须把 RoPE 部分单独拆出来缓存？d_c=512、RoPE 通道 64 时，每层每 token 缓存多少个数？
4. 权重吸收 q·(W·c) = (q·W)·c 为什么成立？吸收后还需要为每个缓存 token 展开每头的 k 吗？
5. SFA 与 2.5 节因果掩码整块跳过，共同思想是什么？本质区别是什么？
6. SFA 在昇腾上的核心工程难题是什么？举两个对策。

> 答案见 `answer/02.09_answer.txt`。

下一章：[第 3 章 FA 算子代码结构](../03_fa_code_architecture/README.md)——原理备齐，去看看真实算子工程的骨架。